[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Mach_Learn/Diffusion_Score_SDE.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Diffusion II: Score Matching & SDEs

The rigorous sequel [Diffusion Models](./Diffusion_Models.ipynb) gestured at: what the network *actually* learns is the **score** $\nabla_x \log p(x)$, the discrete chain is an [SDE](../Intro_Math/Stochastic_Processes/Stochastic_Processes_2.ipynb) in disguise, and generation is that SDE run backwards. Verified on a Gaussian mixture where the true score is available in closed form — the oracle most tutorials never check.

## 1. Pre-requisites

[Diffusion Models](./Diffusion_Models.ipynb) (the practical loop), [Stochastic Processes II](../Intro_Math/Stochastic_Processes/Stochastic_Processes_2.ipynb) S4 (Brownian motion, Itô).

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

# ground-truth distribution: a 2-D Gaussian mixture — score known in CLOSED FORM
centers = torch.tensor([[-2.0, 0.0], [2.0, 0.0], [0.0, 2.2]])
sig2 = 0.35**2
def sample_data(n):
    idx = torch.randint(0, 3, (n,))
    return centers[idx] + 0.35*torch.randn(n, 2)
def true_score(x, t_var=0.0):
    """∇ log p_t for the mixture convolved with N(0, t_var) — exact."""
    v = sig2 + t_var
    d2 = ((x[:, None, :] - centers[None])**2).sum(-1)
    w = torch.softmax(-d2/(2*v), dim=1)
    mu_post = (w[:, :, None] * centers[None]).sum(1)
    return (mu_post - x) / v

**Why a toy mixture.** Three Gaussian blobs are not a demo of what diffusion can *do* — for that, see the images in [Diffusion Models](./Diffusion_Models.ipynb). They are here because a mixture of Gaussians is one of the few distributions whose score we can write down exactly, in the `true_score` function above. Everything in this workshop is a claim of the form "the network learns the score" or "the chain is an SDE," and on this toy problem each claim becomes a number we can check rather than a sentence we have to trust. That is the trade we are making: we give up visual impressiveness and buy an oracle.

---
### 🕐 Session 1 of 3 — *The Score Function* (~40 min)
**Goal:** learn ∇ log p by denoising; verify against the closed-form mixture score.
**Builds on:** [Diffusion Models](./Diffusion_Models.ipynb). &nbsp; **Feeds into:** Session 2 (Langevin & SDEs).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: The Score Function</b></summary>

**Timing (~40 min).** 5 min recap of Diffusion I's training loop · 10 min score intuition at the board · 10 min the denoising-score-matching identity · 12 min the demo · 3 min buffer. If you are late, cut the recap, never the identity — it is the load-bearing idea of the whole workshop.

**Board first.** Draw the three blobs, then have the room call out arrow directions at a few points before you write any formula. Students who *see* the compass field never afterwards confuse the score with "the direction the optimizer moves."

**Misconception — the big one.** "Score = ∇loss." It is the gradient of the *log-density*, taken with respect to $x$, not with respect to the parameters. Ask someone to say aloud which variable we differentiate, and don't move on until the answer is $x$. Almost every downstream confusion in Sessions 2–3 traces back to this.

**Ask the room.** "Why does the normalizing constant disappear?" Let them work it: $\log(p/Z) = \log p - \log Z$, and $\log Z$ is constant in $x$, so it dies under $\nabla_x$. This is the single reason score-based modelling is tractable at all — worth the 90 seconds of silence.

**Prereq check.** If the room looks shaky on $\nabla \log$, do the 1-D Gaussian on the board: $\log p = -\tfrac{(x-\mu)^2}{2\sigma^2} + c$, so $s(x) = (\mu - x)/\sigma^2$ — an arrow pointing back at the mean, with strength proportional to how far out you are. That one line makes `true_score` readable.

**If the demo misbehaves.** Median cosine below ~0.99 nearly always means too few steps; bump `range(3000)` to 5000 and keep talking while it trains. The arrows *should* disagree in the far corners — that is not a bug, and it is worth pointing at (see the debrief below).
</details>

## 2. The Gradient of the Log-Density

💡 **Intuition.** The score $s(x) = \nabla_x \log p(x)$ is a *compass field*: at every point it points toward higher probability. You never need the (intractable) normalizing constant — gradients of $\log p$ kill it. And the miracle that makes it learnable: **denoising score matching** — training a network to predict the noise added to data is, up to scale, training it to output the score of the *noised* distribution ($s = -\varepsilon/\sigma$). Diffusion I's 'predict the noise' loss was secretly score estimation all along. Here we can *prove* it: our mixture's score has a closed form to compare against.

In [2]:
# train a denoiser at ONE noise level; compare its implied score with the exact one
sigma_noise = 0.5
net = nn.Sequential(nn.Linear(2, 128), nn.SiLU(), nn.Linear(128, 128), nn.SiLU(), nn.Linear(128, 2))
opt = torch.optim.Adam(net.parameters(), lr=2e-3)
for step in range(3000):
    x0 = sample_data(512)
    eps = torch.randn_like(x0)
    xt = x0 + sigma_noise*eps
    loss = ((net(xt) - eps)**2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

# ORACLE: implied score −net(x)/σ vs the closed-form mixture score at this noise level
g = torch.linspace(-3.5, 3.5, 24)
GX, GY = torch.meshgrid(g, g, indexing="xy")
pts = torch.stack([GX.ravel(), GY.ravel()], 1)
with torch.no_grad():
    s_learned = -net(pts)/sigma_noise
s_exact = true_score(pts, t_var=sigma_noise**2)
cos = nn.functional.cosine_similarity(s_learned, s_exact, dim=1)
rel = (s_learned - s_exact).norm(dim=1) / (s_exact.norm(dim=1) + 1e-6)
print(f"learned vs exact score: median cosine {cos.median():.4f}, median relative error {rel.median():.3f}")

plt.figure(figsize=(5.2, 4.6))
plt.quiver(GX, GY, s_exact[:,0].reshape(24,24), s_exact[:,1].reshape(24,24), color="gray", alpha=0.6, label="exact")
plt.quiver(GX, GY, s_learned[:,0].reshape(24,24), s_learned[:,1].reshape(24,24), color="crimson", alpha=0.6, scale=None, label="learned")
plt.scatter(*sample_data(400).T, s=2, alpha=0.3)
plt.legend(fontsize=7); plt.title("the compass field, learned from denoising alone")
plt.tight_layout(); plt.show()

learned vs exact score: median cosine 0.9987, median relative error 0.098


/tmp/ipykernel_2995578/43406241.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Median cosine similarity **0.9987**, median relative error **0.098** — the red arrows sit almost exactly on the grey ones. That is the whole claim of this session, measured: the network was trained *only* to predict the noise $\varepsilon$, and never saw `true_score` during training, yet $-\text{net}(x)/\sigma$ reproduces the closed-form score of the noised mixture. "Predict the noise" and "estimate the score" are the same objective wearing different clothes.

Two details worth reading off the plot. The arrows agree tightly wherever the blue data points are, and drift apart in the far corners — the network only ever saw training samples where the data lives, so it has no reason to be right in the tails. And the direction is far more accurate than the magnitude (cosine 0.999 against a ~10% relative error), which is exactly what we want, because the samplers in Sessions 2 and 3 use the score as a *direction to move* and then control the step size themselves.

---
### 🕐 Session 2 of 3 — *Langevin Dynamics & the Forward SDE* (~40 min)
**Goal:** climb the score with noise: Langevin sampling; the diffusion chain as an SDE.
**Builds on:** Session 1; [Stochastic Processes II](../Intro_Math/Stochastic_Processes/Stochastic_Processes_2.ipynb). &nbsp; **Feeds into:** Session 3 (the reverse SDE & probability flow).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Langevin Dynamics & the Forward SDE</b></summary>

**Timing (~40 min).** 10 min Langevin at the board · 10 min why plain Langevin fails on separated modes · 12 min the SDE reframing and the OU audit · 8 min buffer. This session has the most board work and the least code; resist the urge to fill the time with more code.

**Board first.** Write the Langevin update and cover the noise term with your hand. Ask what the algorithm does without it — gradient ascent on $\log p$, so every particle slides to the nearest mode and stops. *Then* uncover the noise. The point lands much harder as a reveal than as a formula presented whole.

**Misconception.** "More noise is worse." Students coming from optimization treat noise purely as an obstacle. Here it is what makes the sampler correct rather than a mode-finder: the $\sqrt{\eta}\,\xi$ term is what makes the stationary distribution $p$ itself instead of $\delta$ at a mode. If someone has seen [SGD's noise](../Intro_Math/Optimization/Optimization.ipynb) framed as a nuisance, name that contrast explicitly.

**Ask the room.** "Our three blobs are well separated — start a particle at the left mode and run Langevin. How long until it visits the right one?" Essentially never at small $\eta$. Let them sit with that, because the answer *is* the design rationale for diffusion: run a family of scores across noise levels, so high noise merges the modes into one broad blob you can travel across, and low noise sharpens the details once you have arrived.

**The reframe to land.** Diffusion I's discrete chain and the OU SDE $dx = -\tfrac12\beta x\,dt + \sqrt{\beta}\,dB$ are the same object at different step sizes. Students often think the SDE is a new, fancier model. It is not — it is the continuum limit of the loop they already wrote last workshop.

**If the demo misbehaves.** The variance curve is a stochastic simulation, so the dots scatter around the black line; a max gap of a few hundredths is the expected result, not a near-miss. If it is much worse, `dt` has been raised — Euler–Maruyama error grows with step size, which is itself a teachable moment.
</details>

## 3. Sampling = Noisy Gradient Ascent

💡 **Intuition.** Given a score, **Langevin dynamics** samples: $x \mathrel{+}= \frac{\eta}{2} s(x) + \sqrt{\eta}\, \xi$ — climb the compass field, but inject just enough noise that you *explore* the distribution instead of collapsing to its modes ([SGD's noise](../Intro_Math/Optimization/Optimization.ipynb), now load-bearing). The catch that motivated diffusion: with far-apart modes, plain Langevin mixes badly — which is why diffusion runs a *family* of scores across noise levels: high noise merges the modes for easy travel, low noise sharpens the details. And in the continuum, Diffusion I's chain **is** the SDE $dx = -\tfrac12\beta x\, dt + \sqrt{\beta}\, dB$ — an Ornstein–Uhlenbeck process whose marginals we can check exactly.

In [3]:
# ORACLE: the forward SDE's variance must follow the OU closed form
beta = 1.2
dt = 0.001
T_steps = 2000
x = sample_data(4000)
var_path = []
for k in range(T_steps):
    x = x - 0.5*beta*x*dt + np.sqrt(beta*dt)*torch.randn_like(x)
    if k % 100 == 0: var_path.append(x.var().item())
t_ax = np.arange(0, T_steps, 100)*dt
var_theory = np.exp(-beta*t_ax)*sample_data(20000).var().item() + (1 - np.exp(-beta*t_ax))
plt.figure(figsize=(7, 2.4))
plt.plot(t_ax, var_path, "o", markersize=4, label="simulated variance")
plt.plot(t_ax, var_theory, "k-", linewidth=1, label="OU closed form")
plt.legend(fontsize=8); plt.xlabel("t"); plt.title("the forward SDE, audited against Itô's answer")
plt.tight_layout(); plt.show()
print(f"max |simulated − OU theory| variance: {np.abs(np.array(var_path)-var_theory).max():.3f}")

max |simulated − OU theory| variance: 0.030


/tmp/ipykernel_2995578/547938826.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** We never solved the SDE — we simulated it, one small Euler–Maruyama step at a time, exactly as Diffusion I's loop does. Itô's calculus says the variance of that process must follow $\mathrm{Var}(t) = e^{-\beta t}\,\mathrm{Var}(0) + (1 - e^{-\beta t})$, and the simulated dots track that closed form to a maximum gap of **0.030**. The chain really is an Ornstein–Uhlenbeck process.

Read the shape, not just the error: the curve starts at the data's own variance and relaxes to 1, no matter what we started with. That limit is why the reverse process in Session 3 can begin from `torch.randn` — the forward SDE deliberately forgets the data and converges to a standard Gaussian we know how to sample. The residual scatter around the black line is Euler–Maruyama discretization error plus finite-sample noise from 4000 particles; it shrinks with smaller `dt`, and it is the same error the reverse sampler will pay.

---
### 🕐 Session 3 of 3 — *The Reverse SDE & Probability Flow* (~40 min)
**Goal:** run time backwards with the score; sample the mixture and audit mode weights.
**Builds on:** Session 2.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: The Reverse SDE & Probability Flow</b></summary>

**Timing (~40 min).** 5 min recap of Sessions 1–2 as two halves of one machine · 10 min Anderson's reverse SDE · 5 min the probability-flow ODE · 15 min the demo (training runs ~1–2 min on CPU) · 5 min buffer. Start the code cell running *before* you discuss the ODE if you want the plot ready on time.

**Recap that sets up the payoff.** Session 1 gave us a way to *learn* $s_t$. Session 2 gave us the forward SDE that destroys data. This session's theorem says those two things are already everything you need to generate — nothing further has to be learned. Framing it that way makes Anderson's result feel earned rather than pulled from a hat.

**Board first.** Write the forward and reverse SDEs one above the other and box the single difference: the $-\beta s_t(x)$ drift term. The entire content of "undoing noise" is that one box, and it is exactly the object Session 1 taught us to estimate. If students take one image away from the workshop, make it this pair of lines.

**Misconception.** "Reversing the SDE means running the simulation backwards in time, like rewinding a video." It does not — a *specific* trajectory is not recovered, and the reverse process is a genuinely different SDE with its own Brownian motion $d\bar{B}$. What is preserved is the sequence of marginal *distributions*, not any individual path. Ask what would happen if you simply negated `dt` in Session 2's loop: the variance would explode rather than shrink.

**Ask the room.** "The probability-flow ODE has no noise at all, yet produces the same marginals. So why would anyone keep the stochastic sampler?" Good answers: the SDE self-corrects, since noise lets a path that has wandered off into a low-density region be pulled back, while the ODE will faithfully follow a wrong trajectory to the end. Good answers the other way: the ODE is deterministic, so it is invertible and gives you exact likelihoods, and it takes far fewer steps — which is precisely why DDIM is its discretization.

**If the demo misbehaves.** Mode weights are a finite-sample estimate over 6000 points, so roughly ±0.02 around 0.333 is normal. Badly imbalanced weights (one mode at 0.5, or a mode missing entirely) mean undertrained — raise `range(6000)`. A cloud that is diffuse rather than in three tight blobs usually means `n_rev` is too small: too few reverse steps means too much discretization error.

**Where to stop.** If time is short, cut the probability-flow ODE discussion, not the reverse-SDE demo. Flow matching gets its own treatment in [Optimal Transport](./Optimal_Transport.ipynb), so the ODE has somewhere else to land; the demo does not.
</details>

## 4. Anderson's Time Machine

💡 **Intuition.** The stunning theorem (Anderson, 1982): the forward SDE has an exact **reverse**: $dx = [-\tfrac12\beta x - \beta\, s_t(x)]\, dt + \sqrt{\beta}\, d\bar{B}$ — identical dynamics plus a score-guided drift. Everything unknown about 'undoing noise' is packed into $s_t$, the exact object Session 1 taught us to learn. Drop the noise term and halve the score drift and you get the **probability-flow ODE** — deterministic, same marginals, the bridge to flow matching and fast samplers (DDIM is its discretization).

In [4]:
# train a time-conditional score net, then sample by reverse SDE — audit the result
class ScoreNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(nn.Linear(2+8, 128), nn.SiLU(), nn.Linear(128, 128), nn.SiLU(), nn.Linear(128, 2))
    def forward(self, x, tt):
        emb = torch.cat([torch.sin(tt[:, None]*torch.arange(1, 5)), torch.cos(tt[:, None]*torch.arange(1, 5))], 1)
        return self.f(torch.cat([x, emb], 1))

T_max = 2.0
score_net = ScoreNet()
opt = torch.optim.Adam(score_net.parameters(), lr=2e-3)
for step in range(6000):
    x0 = sample_data(512)
    tt = torch.rand(512)*T_max + 1e-3
    a = torch.exp(-0.5*beta*tt)[:, None]                       # OU mean decay
    var = (1 - torch.exp(-beta*tt))[:, None]
    eps = torch.randn_like(x0)
    xt = a*x0 + var.sqrt()*eps
    target = -eps/var.sqrt()                                   # the score of the perturbed marginal
    loss = ((score_net(xt, tt) - target)**2 * var).mean()      # standard weighting
    opt.zero_grad(); loss.backward(); opt.step()

# reverse SDE from pure noise
x = torch.randn(6000, 2)
n_rev = 400
dt_r = T_max/n_rev
with torch.no_grad():
    for k in range(n_rev):
        tt = torch.full((len(x),), T_max - k*dt_r)
        s = score_net(x, tt)
        x = x + (0.5*beta*x + beta*s)*dt_r + np.sqrt(beta*dt_r)*torch.randn_like(x)

plt.figure(figsize=(4.6, 4))
plt.scatter(*x.T, s=2, alpha=0.2, label="reverse-SDE samples")
plt.scatter(*centers.T, marker="*", s=150, c="crimson", label="true mode centers")
plt.legend(fontsize=7); plt.axis("equal"); plt.title("noise → mixture, via the learned score")
plt.tight_layout(); plt.show()

# ORACLE: mode weights should be ≈ 1/3 each; mode means ≈ the centers
assign = ((x[:, None, :] - centers[None])**2).sum(-1).argmin(1)
for k in range(3):
    frac = (assign == k).float().mean()
    mu = x[assign == k].mean(0)
    print(f"mode {k}: weight {frac:.3f} (true 0.333)   center {mu.numpy().round(2)} (true {centers[k].numpy()})")

mode 0: weight 0.354 (true 0.333)   center [-2.08  0.05] (true [-2.  0.])
mode 1: weight 0.330 (true 0.333)   center [ 2.06 -0.02] (true [2. 0.])
mode 2: weight 0.316 (true 0.333)   center [0.03 2.19] (true [0.  2.2])


/tmp/ipykernel_2995578/2519242878.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** We started from pure Gaussian noise — no data whatsoever — and ran Anderson's reverse SDE for 400 steps using nothing but the learned score. Out came the mixture: weights **0.354 / 0.330 / 0.316** against a true 1/3 each, with recovered centers $(-2.08, 0.05)$, $(2.06, -0.02)$, $(0.03, 2.19)$ against the true $(-2, 0)$, $(2, 0)$, $(0, 2.2)$.

The weights matter more than the centers here, and it is worth saying why. Getting the centers right only shows the sampler finds the modes — a mode-seeking method that ignored the noise term would manage that too. Getting the *weights* right shows it visits them in the correct proportion, which is the difference between a sampler and an optimizer. That is the property Session 2's noise term buys, now measured end to end.

Note also what this run did *not* require: no Metropolis correction, no annealing schedule to tune by hand, no separate model per noise level. One time-conditional score network, plus a theorem from 1982, and generation falls out. The residual few percent in the weights is the accumulation of everything we have already accounted for — score error in the tails from Session 1, discretization error from Session 2, and finite-sample noise from 6000 points.

## 5. Conclusion

'Predict the noise' is score estimation (verified against a closed-form score, cosine ≈ 1); the chain is an OU SDE (variance audited against Itô); and Anderson's reverse SDE turns the learned compass into a sampler whose mode weights and centers match the truth. Diffusion I's recipe now has its complete mathematical spine.

---
## Where next

- [Stochastic Processes II](../Intro_Math/Stochastic_Processes/Stochastic_Processes_2.ipynb) — the Itô calculus underneath.
- [Optimal Transport](./Optimal_Transport.ipynb) — probability flow as a transport map; flow matching lives here.